# Distance Threshold Exceedance Analysis

## Proportion of SALs Beyond Pharmacy Access Thresholds (k = 1, 2, 3)

**Tess Vu**

Binary accessibility screening: for each SAL, does a pharmacy exist
within a given network distance threshold? This complements the 2SFCA by
providing a policy-legible metric that avoids decay functions,
demand-weighting, and provincial quantiles. The question is simple:
**can people physically reach medication or not?**

Three levels of k are analyzed separately:

- **k = 1 (reachability):** Can a person reach *any* pharmacy within the
  threshold? This is the minimum viability question.
- **k = 2 (redundancy):** Is there a second pharmacy within reach? If
  the nearest pharmacy is closed, stocked out, or overcrowded, is there
  a fallback?
- **k = 3 (choice):** Do residents have meaningful options? Three
  reachable pharmacies suggests a functioning local market rather than a
  single point of dependency.

Thresholds are applied to pre-computed k-nearest-pharmacy distances
(Euclidean, pedestrian network, drive network) from SAL centroids,
produced by the `sal_pharmacy_distance_k3` notebook.

## SETUP AND IMPORTS

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.font_manager as fm
import warnings

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

In [ ]:
import sys
from pathlib import Path

ROOT = Path("..").resolve()          # repository root; notebooks run from notebooks/
DATA = ROOT / "data"
RAW = DATA / "raw"
INTERMEDIATE = DATA / "intermediate"
FINAL = DATA / "final"

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import dair_style as ds
IMAGES = Path("images") / "network_threshold"

FONT_TITLE = ds.FONT_TITLE
FONT_BODY = ds.FONT_BODY

IN_DISTANCES = INTERMEDIATE / "sal_pharmacy_distances_k3.csv"
IN_POP_PRED = INTERMEDIATE / "pop_pred_final.csv"
OUT_THRESHOLD_FLAGS = INTERMEDIATE / "sal_threshold_flags_k3.csv"
OUT_THRESHOLD_SUMMARY = INTERMEDIATE / "threshold_summary_k3.csv"


## CONFIGURATION

All file paths and threshold definitions in a single cell. Thresholds
are set once and used throughout the notebook.

In [ ]:
# ANALYSIS CONFIGURATION

# K LEVELS TO ANALYZE
K_LEVELS = [1, 2, 3]

# DISTANCE THRESHOLDS (kilometers)
# Walking thresholds reflect pedestrian-realistic access bands.
WALK_THRESHOLDS_KM = [1, 2, 3, 5]

# Driving thresholds reflect vehicle-based access bands.
DRIVE_THRESHOLDS_KM = [5, 10, 15, 20]

# Euclidean thresholds for straight-line baseline comparison.
EUCLIDEAN_THRESHOLDS_KM = [1, 3, 5, 10]

# PROVINCE LOOKUP from EA_CODE first digit.
# Stats SA geographic coding: 5 = KwaZulu-Natal, 7 = Gauteng.
PROVINCE_MAP = {
    "5": "KwaZulu-Natal",
    "7": "Gauteng"
}

# Geography Type display order.
SETTLEMENT_ORDER = ["Urban", "Traditional", "Farms"]


## LOAD DATA

In [ ]:
# Load pre-computed SAL-to-pharmacy distances (k = 3 version).
dist_raw = pd.read_csv(IN_DISTANCES)
print(f"DISTANCE DATA LOADED: {dist_raw.shape[0]} rows, {dist_raw.shape[1]} columns")
print(f"Columns: {list(dist_raw.columns)}")
print()

# Load population and geography type data.
pop_raw = pd.read_csv(IN_POP_PRED)
print(f"POPULATION DATA LOADED: {pop_raw.shape[0]} rows, {pop_raw.shape[1]} columns")
print(f"Columns: {list(pop_raw.columns)}")

## MERGE AND PREPARE ANALYSIS TABLE

Join distances with population attributes (geography type, economic
status, estimated population). Province is derived from the EA_CODE
prefix as a fallback if not present in the distance file.

In [ ]:
# Normalize EA_CODE to int64 on both sides.
pop_raw["EA_CODE"] = pd.to_numeric(pop_raw["EA_CODE"], errors = "coerce").astype("Int64")

# Select relevant columns from population data.
pop_cols = ["EA_CODE", "EA_GTYPE", "EA_TYPE", "econ_status", "sal2023_est", "area_km2"]
pop_subset = pop_raw[pop_cols].copy()

# Drop sal2023_est from distance data if it exists (prefer population version).
dist_raw = dist_raw.drop(columns = ["sal2023_est"], errors = "ignore")

# Merge distance data with population attributes.
df = dist_raw.merge(pop_subset, on = "EA_CODE", how = "left")

print(f"MERGED SHAPE: {df.shape}")
print(f"Match rate: {df['EA_GTYPE'].notna().sum()} / {len(df)} ({df['EA_GTYPE'].notna().mean() * 100:.1f}%)")

In [ ]:
# Derive province from EA_CODE if not already present.
if "PR_NAME" not in df.columns:
    df["PR_NAME"] = df["EA_CODE"].astype(str).str[0].map(PROVINCE_MAP)
    print("PROVINCE DERIVED FROM EA_CODE PREFIX")
else:
    print("PROVINCE COLUMN ALREADY PRESENT IN DISTANCE DATA")

print(f"Province distribution:")
print(df["PR_NAME"].value_counts())
print()

# Confirm geography type distribution.
print("GEOGRAPHY TYPE (EA_GTYPE) DISTRIBUTION:")
print(df["EA_GTYPE"].value_counts())
print()

# Drop rows with missing province or geography type.
before = len(df)
df = df.dropna(subset = ["PR_NAME", "EA_GTYPE"]).copy()
after = len(df)
print(f"ROWS DROPPED (missing province or geography type): {before - after}")
print(f"ANALYSIS TABLE: {after} SALs")

In [ ]:
# Identify available distance columns for each k level.
# Column naming convention: {mode}_dist_k{ki}_km.
dist_col_map = {}

for ki in K_LEVELS:
    mode_cols = {}
    for mode, col_pattern in [("walk", f"walk_dist_k{ki}_km"),
                               ("drive", f"drive_dist_k{ki}_km"),
                               ("euclidean", f"euclidean_dist_k{ki}_km")]:
        if col_pattern in df.columns:
            mode_cols[mode] = col_pattern
            valid = df[col_pattern].notna().sum()
            print(f"  k={ki} {mode}: column '{col_pattern}' found, {valid} valid values ({valid / len(df) * 100:.1f}%)")
        else:
            print(f"  k={ki} {mode}: column '{col_pattern}' NOT FOUND")
    dist_col_map[ki] = mode_cols

print()
print(f"K LEVELS WITH DATA: {list(dist_col_map.keys())}")

## DISTANCE OVERVIEW BY PROVINCE AND GEOGRAPHY TYPE

Quick summary statistics before threshold analysis. Reported for k=1
(nearest pharmacy) to establish baseline, with k=2 and k=3 medians
for comparison.

In [ ]:
# Summary statistics for k=1 distances by province.
k1_modes = dist_col_map.get(1, {})

for mode, col in k1_modes.items():
    print(f"DISTANCE SUMMARY (k=1): {mode.upper()} (km)")
    summary = df.groupby("PR_NAME")[col].describe(percentiles = [0.25, 0.5, 0.75, 0.9, 0.95])
    print(summary.round(2).to_string())
    print()

In [ ]:
# Median distance comparison across k levels by province and geography type.
print("MEDIAN DISTANCE (km) ACROSS K LEVELS BY PROVINCE x GEOGRAPHY TYPE")
print()

for mode in ["walk", "drive"]:
    print(f"  {mode.upper()} NETWORK")
    rows = []
    for ki in K_LEVELS:
        col = dist_col_map.get(ki, {}).get(mode)
        if col is None:
            continue
        pivot = df.pivot_table(
            values = col,
            index = "EA_GTYPE",
            columns = "PR_NAME",
            aggfunc = "median"
        )
        pivot = pivot.reindex(SETTLEMENT_ORDER)
        for gtype in SETTLEMENT_ORDER:
            for province in ["Gauteng", "KwaZulu-Natal"]:
                val = pivot.loc[gtype, province] if gtype in pivot.index and province in pivot.columns else np.nan
                rows.append({
                    "Geography": gtype,
                    "Province": province,
                    "k": ki,
                    "Median (km)": round(val, 2) if pd.notna(val) else np.nan
                })

    comparison = pd.DataFrame(rows)
    comparison_pivot = comparison.pivot_table(
        values = "Median (km)",
        index = ["Province", "Geography"],
        columns = "k"
    )
    print(comparison_pivot.round(2).to_string())
    print()

## THRESHOLD EXCEEDANCE FLAGS

For each SAL, each threshold, and each k level, flag whether the k-th
nearest pharmacy distance exceeds the threshold. A value of `True` means
the SAL **fails** the accessibility test at that threshold for that k.

For k=1 a flag of `True` means no pharmacy is within reach. For k=2 or
k=3 a flag of `True` means the SAL does not have two or three pharmacies
within reach.

In [ ]:
# Map thresholds to distance modes.
threshold_config = {
    "walk": WALK_THRESHOLDS_KM,
    "drive": DRIVE_THRESHOLDS_KM,
    "euclidean": EUCLIDEAN_THRESHOLDS_KM
}

# Create binary exceedance flags for each k level.
flag_columns = {}

for ki in K_LEVELS:
    ki_flags = []
    modes = dist_col_map.get(ki, {})

    for mode, col in modes.items():
        thresholds = threshold_config.get(mode, [])
        for t in thresholds:
            flag_col = f"exceeds_{mode}_k{ki}_{t}km"
            df[flag_col] = df[col] > t
            ki_flags.append(flag_col)

    flag_columns[ki] = ki_flags
    print(f"K={ki}: {len(ki_flags)} FLAGS CREATED")

print()

# Print overall exceedance summary for k=1.
print("K=1 EXCEEDANCE SUMMARY (all SALs)")
for fc in flag_columns.get(1, []):
    n_exceed = df[fc].sum()
    print(f"  {fc}: {n_exceed} SALs ({n_exceed / len(df) * 100:.1f}%)")

## EXCEEDANCE PROPORTIONS BY PROVINCE (k = 1, 2, 3)

Proportion of SALs exceeding each threshold, split by province. Both
unweighted (SAL count) and population-weighted proportions are computed.

The population-weighted version answers: what share of *people* live
beyond the threshold for their k-th nearest pharmacy?

In [ ]:
def compute_exceedance_table(data, flag_cols, group_col, weight_col = None):
    """Compute proportion exceeding threshold, grouped by a column.

    Parameters
    data : DataFrame with flag columns and grouping column.
    flag_cols : list of boolean flag column names.
    group_col : column to group by.
    weight_col : if provided, compute population-weighted proportions.

    Returns
    DataFrame with groups as rows and thresholds as columns.
    """
    results = {}
    for grp, grp_df in data.groupby(group_col):
        row = {}
        for fc in flag_cols:
            if weight_col and weight_col in grp_df.columns:
                total_pop = grp_df[weight_col].sum()
                exceed_pop = grp_df.loc[grp_df[fc], weight_col].sum()
                row[fc] = exceed_pop / total_pop if total_pop > 0 else np.nan
            else:
                row[fc] = grp_df[fc].mean()
        results[grp] = row
    return pd.DataFrame(results).T

In [ ]:
# UNWEIGHTED EXCEEDANCE by province for each k level.
for ki in K_LEVELS:
    print(f"UNWEIGHTED EXCEEDANCE PROPORTIONS BY PROVINCE (k = {ki}, % of SALs)")
    print()

    ki_modes = dist_col_map.get(ki, {})
    ki_flags = flag_columns.get(ki, [])

    for mode in ["walk", "drive", "euclidean"]:
        mode_flags = [f for f in ki_flags if f.startswith(f"exceeds_{mode}_k{ki}_")]
        if not mode_flags:
            continue
        table = compute_exceedance_table(df, mode_flags, "PR_NAME")
        display_cols = {f: f.replace(f"exceeds_{mode}_k{ki}_", "").replace("km", " km") for f in mode_flags}
        table = table.rename(columns = display_cols)
        print(f"  {mode.upper()} NETWORK")
        print((table * 100).round(1).to_string())
        print()
    print()

In [ ]:
# POPULATION-WEIGHTED EXCEEDANCE by province for each k level.
for ki in K_LEVELS:
    print(f"POPULATION-WEIGHTED EXCEEDANCE BY PROVINCE (k = {ki}, % of people)")
    print()

    ki_flags = flag_columns.get(ki, [])

    for mode in ["walk", "drive", "euclidean"]:
        mode_flags = [f for f in ki_flags if f.startswith(f"exceeds_{mode}_k{ki}_")]
        if not mode_flags:
            continue
        table = compute_exceedance_table(df, mode_flags, "PR_NAME", weight_col = "sal2023_est")
        display_cols = {f: f.replace(f"exceeds_{mode}_k{ki}_", "").replace("km", " km") for f in mode_flags}
        table = table.rename(columns = display_cols)
        print(f"  {mode.upper()} NETWORK")
        print((table * 100).round(1).to_string())
        print()
    print()

## EXCEEDANCE BY PROVINCE AND GEOGRAPHY TYPE

The critical cross-tabulation: how do threshold exceedance rates vary
across geography types within each province? Traditional and farm
settlements in KZN are expected to show the highest exceedance rates,
reflecting inherited apartheid-era infrastructure deficits.

Reported for k=1 (reachability) and k=3 (choice) side by side.

In [ ]:
# Cross-tabulation: province x geography type, for k=1 and k=3.
for ki in [1, 3]:
    print(f"EXCEEDANCE PROPORTIONS BY PROVINCE x GEOGRAPHY TYPE (k = {ki}, % of SALs)")
    print()

    ki_flags = flag_columns.get(ki, [])

    for mode in ["walk", "drive"]:
        mode_flags = [f for f in ki_flags if f.startswith(f"exceeds_{mode}_k{ki}_")]
        if not mode_flags:
            continue

        print(f"  {mode.upper()} NETWORK")

        for province in ["Gauteng", "KwaZulu-Natal"]:
            prov_df = df[df["PR_NAME"] == province]
            table = compute_exceedance_table(prov_df, mode_flags, "EA_GTYPE")
            table = table.reindex(SETTLEMENT_ORDER).dropna(how = "all")
            display_cols = {f: f.replace(f"exceeds_{mode}_k{ki}_", "").replace("km", " km") for f in mode_flags}
            table = table.rename(columns = display_cols)
            print(f"    {province}")
            print(f"    n = {len(prov_df)}")
            print((table * 100).round(1).to_string())
            print()
    print()

## REDUNDANCY GAP: k=1 vs k=3

This section directly compares the k=1 and k=3 exceedance rates at the
same threshold. The difference reveals how many SALs pass the basic
reachability test (k=1) but fail the choice test (k=3). These are SALs
with fragile access, dependent on a single pharmacy.

In [ ]:
# Compute the redundancy gap for walk and drive at key thresholds.
print("REDUNDANCY GAP: % SALs EXCEEDING THRESHOLD AT k=3 MINUS k=1")
print("(Positive values = SALs with access to 1 pharmacy but not 3)")
print()

key_thresholds = {
    "walk": [3, 5],
    "drive": [10, 20]
}

for mode, thresholds in key_thresholds.items():
    print(f"  {mode.upper()} NETWORK")
    for t in thresholds:
        k1_flag = f"exceeds_{mode}_k1_{t}km"
        k3_flag = f"exceeds_{mode}_k3_{t}km"

        if k1_flag not in df.columns or k3_flag not in df.columns:
            print(f"    {t} km: columns not available")
            continue

        for province in ["Gauteng", "KwaZulu-Natal"]:
            prov_df = df[df["PR_NAME"] == province]
            n = len(prov_df)
            k1_pct = prov_df[k1_flag].mean() * 100
            k3_pct = prov_df[k3_flag].mean() * 100
            gap = k3_pct - k1_pct
            print(f"    {province} @ {t} km: k1 = {k1_pct:.1f}%, k3 = {k3_pct:.1f}%, gap = {gap:.1f} pp")

    print()

## VISUALIZATIONS

Grouped bar charts comparing k=1 vs k=3 exceedance rates side by side,
highlighting the redundancy gap.

In [ ]:
# Grouped bar chart: Walk exceedance by province, k=1 vs k=3.
fig, axes = plt.subplots(1, 2, figsize = (14, 5), sharey = True)

for ax_idx, province in enumerate(["Gauteng", "KwaZulu-Natal"]):
    prov_df = df[df["PR_NAME"] == province]
    n_prov = len(prov_df)

    k_vals = [1, 3]
    x = np.arange(len(WALK_THRESHOLDS_KM))
    width = 0.35
    colors = [ds.NAVY, ds.YELLOW]

    for j, ki in enumerate(k_vals):
        pcts = []
        for t in WALK_THRESHOLDS_KM:
            flag = f"exceeds_walk_k{ki}_{t}km"
            if flag in prov_df.columns:
                pcts.append(prov_df[flag].mean() * 100)
            else:
                pcts.append(0)

        offset = (j - 0.5) * width
        bars = axes[ax_idx].bar(x + offset, pcts, width, label = f"k = {ki}", color = colors[j])

        # Add value labels.
        for bar, pct in zip(bars, pcts):
            if pct > 3:
                axes[ax_idx].text(
                    bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                    f"{pct:.0f}%", ha = "center", va = "bottom", fontsize = 8
                )

    axes[ax_idx].set_xlabel("Walk Threshold (km)")
    axes[ax_idx].set_xticks(x)
    axes[ax_idx].set_xticklabels([f"{t}" for t in WALK_THRESHOLDS_KM])
    axes[ax_idx].set_title(f"{province} (n = {n_prov:,})", fontsize = 12)
    axes[ax_idx].legend()
    axes[ax_idx].set_ylim(0, 105)

axes[0].set_ylabel("% SALs Exceeding Threshold")
fig.suptitle("Walk Network: k=1 (Reachability) vs k=3 (Choice)",
             fontsize = 14)
plt.tight_layout()
ds.save_fig("walk-k1-vs-k3-bars", fig, fig_dir = IMAGES)

In [ ]:
# Grouped bar chart: Drive exceedance by province, k=1 vs k=3.
fig, axes = plt.subplots(1, 2, figsize = (14, 5), sharey = True)

for ax_idx, province in enumerate(["Gauteng", "KwaZulu-Natal"]):
    prov_df = df[df["PR_NAME"] == province]
    n_prov = len(prov_df)

    k_vals = [1, 3]
    x = np.arange(len(DRIVE_THRESHOLDS_KM))
    width = 0.35
    colors = [ds.NAVY, ds.YELLOW]

    for j, ki in enumerate(k_vals):
        pcts = []
        for t in DRIVE_THRESHOLDS_KM:
            flag = f"exceeds_drive_k{ki}_{t}km"
            if flag in prov_df.columns:
                pcts.append(prov_df[flag].mean() * 100)
            else:
                pcts.append(0)

        offset = (j - 0.5) * width
        bars = axes[ax_idx].bar(x + offset, pcts, width, label = f"k = {ki}", color = colors[j])

        for bar, pct in zip(bars, pcts):
            if pct > 3:
                axes[ax_idx].text(
                    bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                    f"{pct:.0f}%", ha = "center", va = "bottom", fontsize = 8
                )

    axes[ax_idx].set_xlabel("Drive Threshold (km)")
    axes[ax_idx].set_xticks(x)
    axes[ax_idx].set_xticklabels([f"{t}" for t in DRIVE_THRESHOLDS_KM])
    axes[ax_idx].set_title(f"{province} (n = {n_prov:,})", fontsize = 12)
    axes[ax_idx].legend()
    axes[ax_idx].set_ylim(0, 105)

axes[0].set_ylabel("% SALs Exceeding Threshold")
fig.suptitle("Drive Network: k=1 (Reachability) vs k=3 (Choice)",
             fontsize = 14)
plt.tight_layout()
ds.save_fig("drive-k1-vs-k3-bars", fig, fig_dir = IMAGES)

## HEATMAPS: GEOGRAPHY TYPE x THRESHOLD

Heatmaps for walk and drive networks showing exceedance rate by
province-geography type combination. Separate panels for k=1 and k=3.

In [ ]:
# Build heatmap data for walk and drive, k=1 and k=3.
for mode, thresholds in [("walk", WALK_THRESHOLDS_KM), ("drive", DRIVE_THRESHOLDS_KM)]:
    fig, axes = plt.subplots(1, 2, figsize = (16, 5), sharey = True, constrained_layout = True)

    for ax_idx, ki in enumerate([1, 3]):
        ki_flags = flag_columns.get(ki, [])
        mode_flags = [f for f in ki_flags if f.startswith(f"exceeds_{mode}_k{ki}_")]
        if not mode_flags:
            continue

        # Build matrix: rows = province/geography type, columns = thresholds.
        heatmap_rows = []
        row_labels = []

        for province in ["Gauteng", "KwaZulu-Natal"]:
            prov_df = df[df["PR_NAME"] == province]
            for gtype in SETTLEMENT_ORDER:
                subset = prov_df[prov_df["EA_GTYPE"] == gtype]
                if len(subset) == 0:
                    continue
                row_labels.append(f"{province} / {gtype}")
                row_vals = []
                for fc in mode_flags:
                    row_vals.append(subset[fc].mean() * 100)
                heatmap_rows.append(row_vals)

        matrix = np.array(heatmap_rows)
        col_labels = [f"{t} km" for t in thresholds]

        im = axes[ax_idx].imshow(matrix, cmap = ds.SHARE_CMAP, aspect = "auto", vmin = 0, vmax = 100)

        axes[ax_idx].set_xticks(range(len(col_labels)))
        axes[ax_idx].set_xticklabels(col_labels)
        axes[ax_idx].set_yticks(range(len(row_labels)))
        axes[ax_idx].set_yticklabels(row_labels)

        # Annotate cells.
        for i in range(matrix.shape[0]):
            for j in range(matrix.shape[1]):
                val = matrix[i, j]
                color = "white" if val > 60 else "black"
                axes[ax_idx].text(j, i, f"{val:.0f}%", ha = "center", va = "center",
                                  color = color, fontsize = 10)

        axes[ax_idx].set_title(f"k = {ki}", fontsize = 12)

    fig.suptitle(f"{mode.capitalize()} Network: % SALs Exceeding Threshold",
                 fontsize = 14)
    cbar = fig.colorbar(im, ax = axes[-1], shrink = 0.8)
    cbar.set_label("% Exceeding")

    ds.save_fig(f"heatmap-{mode}-k1-k3", fig, fig_dir = IMAGES)

## POLICY SUMMARY TABLE

Formatted output for direct use in reports. Each row answers: at
threshold X, what proportion of SALs (and people) in each province
cannot reach their k-th nearest pharmacy?

In [ ]:
# Build a clean summary table for report use.
summary_rows = []

for ki in K_LEVELS:
    ki_modes = dist_col_map.get(ki, {})

    for mode in ["walk", "drive"]:
        col = ki_modes.get(mode)
        if col is None:
            continue
        thresholds = threshold_config[mode]

        for t in thresholds:
            flag_col = f"exceeds_{mode}_k{ki}_{t}km"
            if flag_col not in df.columns:
                continue

            for province in ["Gauteng", "KwaZulu-Natal"]:
                prov_df = df[df["PR_NAME"] == province]
                n_total = len(prov_df)
                n_exceed = prov_df[flag_col].sum()
                pct_sal = n_exceed / n_total * 100

                pop_total = prov_df["sal2023_est"].sum()
                pop_exceed = prov_df.loc[prov_df[flag_col], "sal2023_est"].sum()
                pct_pop = pop_exceed / pop_total * 100 if pop_total > 0 else np.nan

                summary_rows.append({
                    "k": ki,
                    "Mode": mode.capitalize(),
                    "Threshold (km)": t,
                    "Province": province,
                    "SALs Total": n_total,
                    "SALs Exceeding": int(n_exceed),
                    "% SALs": round(pct_sal, 1),
                    "Pop Exceeding": int(pop_exceed),
                    "% Pop": round(pct_pop, 1)
                })

summary_df = pd.DataFrame(summary_rows)
print("POLICY SUMMARY TABLE")
print(summary_df.to_string(index = False))

## EXPORT RESULTS

In [ ]:
# Export the analysis table with all threshold flags.
export_cols = ["EA_CODE", "PR_NAME", "EA_GTYPE", "EA_TYPE", "econ_status", "sal2023_est"]

# Add distance columns for all k levels.
for ki in K_LEVELS:
    for mode, col in dist_col_map.get(ki, {}).items():
        export_cols.append(col)

# Add all flag columns.
for ki in K_LEVELS:
    export_cols.extend(flag_columns.get(ki, []))

# Filter to columns that exist.
export_cols = [c for c in export_cols if c in df.columns]

# Export full analysis table.
df[export_cols].to_csv(OUT_THRESHOLD_FLAGS, index = False)
print(f"EXPORTED: {OUT_THRESHOLD_FLAGS}")
print(f"Rows: {len(df)}, Columns: {len(export_cols)}")
print()

# Export the summary table.
summary_df.to_csv(OUT_THRESHOLD_SUMMARY, index = False)
print(f"EXPORTED: {OUT_THRESHOLD_SUMMARY}")
